# Deep Learning PR2 — Adult Income Prediction

## Comparative Study of Deep Learning Techniques

### Objective
The objective of this project is to predict whether an individual's annual income is:

- <=50K
- >50K

using the Adult Income (Census Income) dataset.

### Experiments
1. Data Loading, Cleaning and EDA
2. Baseline Artificial Neural Network
3. Activation Function Comparison
4. Weight Initialization Comparison
5. Loss Function Comparison
6. Batch Normalization Comparison
7. Optimizer Comparison
8. Final Best Model
9. Complete Results Comparison

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

In [ ]:
import numpy as np
import pandas as pd
import random
import tensorflow as tf

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

In [ ]:
DATA_PATH = "dataset/adult.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("rows:", df.shape[0] )
print("columns:", df.shape[1] )

print("Missing values per column:")
print(df.isnull().sum())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset info:")
df.info()


In [ ]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

In [ ]:
#remove leading and trailing spaces from string columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

#convert ? to Nan

df = df.replace('?', np.nan)
print("\nMissing values per column after replacing '?' with NaN:")
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values per column after filling NaN in categorical columns:")
print(df.isnull().sum().sum())



In [ ]:
columns_to_drop = []

if "fnlwgt" in df.columns:
    columns_to_drop.append("fnlwgt")

if "education.num" in df.columns:
    columns_to_drop.append("education.num")

df = df.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop)

print("\nDataset shape after dropping columns:", df.shape)


In [ ]:
target_column = ["income", "Income"]  # Assuming 'income' is the target column

target_col = None
for col in target_column:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found in the dataset.")

print("Target column:" ,target_col)

print(df[target_col].value_counts())


In [ ]:
plt.figure(figsize=(7, 5), facecolor="#070808")

ax = sns.countplot(
    x=df[target_col],
    hue=df[target_col],
    palette=["skyblue", "orange"],
    legend=False
)

ax.set_facecolor("#070808")

plt.title("Income Class Distribution", color="#FFFFFF")
plt.xlabel("Income Class", color="#FFFFFF")
plt.ylabel("Count", color="#FFFFFF")

plt.xticks([0, 1], ["<=50K", ">50K"], color="#FFFFFF")
plt.yticks(color="#FFFFFF")

plt.show()

### Class Distribution

The dataset is imbalanced, with more samples in the `<=50K` class than the `>50K` class.

In [ ]:
class_percentages = df[target_col].value_counts(normalize=True).sort_index() * 100

print("\nClass distribution percentages:")
print(class_percentages)

print("\nClass 0 (<=50K): {:.2f}%".format(class_percentages.get(0,0)))
print("Class 1 (>50K): {:.2f}%".format(class_percentages.get(1,0)))

In [ ]:
x = df.drop(columns=[target_col])
y = df[target_col]

print("\nFeatures shape:", x.shape)
print("y shape:", y.shape)

print("\nNumerical columns:")
print(x.select_dtypes(include=['number']).columns.tolist())

print("\nCategorical columns:")
print(x.select_dtypes(include=['object']).columns.tolist())

In [ ]:
X_encoded = pd.get_dummies(
    x, 
    drop_first=True,
    dtype=float)

print("\nEncoded features shape:", X_encoded.shape)
display(X_encoded.head())   


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("\nTraining set shape:", x_train.shape, y_train.shape)
print("Test set shape:", x_test.shape, y_test.shape)

print("\nTraining set class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest set class distribution:")
print(y_test.value_counts(normalize=True))


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

print("Scaling completed")

print("Scaled training set shape:", X_train_scaled.shape)
print("Scaled test set shape:", X_test_scaled.shape)



In [ ]:
# Convert target labels to numeric values
y_train = y_train.replace({'<=50K': 0, '>50K': 1})
y_test = y_test.replace({'<=50K': 0, '>50K': 1})

# Convert to NumPy arrays
y_train = np.asarray(y_train, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

# Convert X data to float32
X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
X_test_scaled = np.asarray(X_test_scaled, dtype=np.float32)

# Check data types
print(X_train_scaled.dtype)
print(X_test_scaled.dtype)
print(y_train.dtype)
print(y_test.dtype)

In [ ]:
classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

print("Class weights:")
print(class_weights)

In [ ]:
def evaluate_model(model, X_data, y_true, model_name="Model"):
    
    y_probability = model.predict(
        X_data,
        verbose=0
    ).ravel()
    
    y_prediction = (y_probability >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, y_prediction)
    precision = precision_score(
        y_true,
        y_prediction,
        zero_division=0
    )
    recall = recall_score(
        y_true,
        y_prediction,
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_prediction,
        zero_division=0
    )
    roc_auc = roc_auc_score(
        y_true,
        y_probability
    )

    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    }

    return results, y_probability, y_prediction

In [ ]:
def build_ann(
    input_dim,
    activation="relu",
    initializer="he_normal",
    optimizer="adam",
    loss="binary_crossentropy",
    batch_norm=False
):
    
    model = keras.Sequential()
    
    model.add(
        layers.Input(shape=(input_dim,))
    )
    
    model.add(
        layers.Dense(
            128,
            kernel_initializer=initializer
        )
    )
    
    if batch_norm:
        model.add(layers.BatchNormalization())
    
    model.add(
        layers.Activation(activation)
    )
    
    model.add(
        layers.Dense(
            64,
            kernel_initializer=initializer
        )
    )
    
    if batch_norm:
        model.add(layers.BatchNormalization())
    
    model.add(
        layers.Activation(activation)
    )
    
    model.add(
        layers.Dense(
            1,
            activation="sigmoid"
        )
    )
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=["accuracy"]
    )
    
    return model

# Task 2 — Baseline ANN

The baseline ANN provides a reference model against which all later experiments will be compared.

Configuration:

- Hidden layers: 128 and 64 neurons
- Hidden activation: ReLU
- Output activation: Sigmoid
- Loss: Binary Cross Entropy
- Optimizer: Adam

In [ ]:
input_dim = X_train_scaled.shape[1]

baseline_model = build_ann(
    input_dim=input_dim,
    activation="relu",
    initializer="he_normal",
    optimizer="adam",
    loss="binary_crossentropy",
    batch_norm=False
)

baseline_model.summary()

In [ ]:
baseline_history = baseline_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=256,
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

plt.plot(
    baseline_history.history["accuracy"],
    label="Training Accuracy",
    linewidth=2
)

plt.plot(
    baseline_history.history["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2
)

plt.title("Baseline ANN — Accuracy", color="#FFFFFF")
plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

plt.legend(facecolor="#070808", labelcolor="#FFFFFF")

plt.show()

### Baseline ANN Accuracy

The model's training and validation accuracy improve as the number of epochs increases.

In [ ]:
plt.figure(figsize=(8, 5), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

plt.plot(
    baseline_history.history["loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    baseline_history.history["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.title("Baseline ANN — Loss", color="#FFFFFF")
plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Loss", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Baseline ANN Loss

Training and validation loss are compared across epochs.

In [ ]:
baseline_result, baseline_prob, baseline_pred = evaluate_model(
    baseline_model,
    X_test_scaled,
    y_test,
    "Baseline ANN"
)

baseline_result

In [ ]:
cm = confusion_matrix(y_test, baseline_pred)

plt.figure(figsize=(6, 5), facecolor="#070808")

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    annot_kws={"color": "white"},
    linewidths=1,
    linecolor="#FFFFFF"
)

plt.title("Baseline ANN — Confusion Matrix", color="#FFFFFF")
plt.xlabel("Predicted", color="#FFFFFF")
plt.ylabel("Actual", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

plt.show()

In [ ]:
print(
    classification_report(
        y_test,
        baseline_pred,
        target_names=["<=50K", ">50K"],
        zero_division=0
    )
)

# Task 3 — Activation Function Comparison

Four activation functions are compared:

1. ReLU
2. Tanh
3. Sigmoid
4. ELU

The purpose is to study how the activation function affects learning,
validation performance and convergence.

In [ ]:
activation_functions = [
    "relu",
    "tanh",
    "sigmoid",
    "elu"
]

activation_results = []
activation_histories = {}

for activation in activation_functions:
    
    print("\n" + "=" * 60)
    print("Training:", activation)
    print("=" * 60)
    
    model = build_ann(
        input_dim=input_dim,
        activation=activation,
        initializer="he_normal",
        optimizer="adam",
        loss="binary_crossentropy",
        batch_norm=False
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        f"Activation - {activation}"
    )
    
    activation_results.append(result)
    activation_histories[activation] = history

In [ ]:
activation_results_df = pd.DataFrame(
    activation_results
)

display(
    activation_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(9, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

for activation, history in activation_histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=activation,
        linewidth=2
    )

plt.title(
    "Activation Function Comparison",
    color="#FFFFFF"
)

plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Validation Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Activation Function Comparison

Validation accuracy is compared for ReLU, Tanh, Sigmoid, and ELU.

# Task 4 — Weight Initialization Comparison

The following initialization techniques are compared:

- Zeros
- Random Normal
- He Normal
- Glorot Uniform

The objective is to study how the initial distribution of weights
affects convergence and learning.

In [ ]:
initializers = {
    "Zeros": "zeros",
    "Random Normal": "random_normal",
    "He Normal": "he_normal",
    "Glorot Uniform": "glorot_uniform"
}

initialization_results = []
initialization_histories = {}

for name, initializer in initializers.items():
    
    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)
    
    model = build_ann(
        input_dim=input_dim,
        activation="relu",
        initializer=initializer,
        optimizer="adam",
        loss="binary_crossentropy",
        batch_norm=False
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        f"Initialization - {name}"
    )
    
    initialization_results.append(result)
    initialization_histories[name] = history

In [ ]:
initialization_results_df = pd.DataFrame(
    initialization_results
)

display(
    initialization_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(10, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

for name, history in initialization_histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=name,
        linewidth=2
    )

plt.title(
    "Weight Initialization — Validation Accuracy",
    color="#FFFFFF"
)

plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Validation Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Weight Initialization Comparison

Validation accuracy is compared for different weight initialization methods.

# Task 5 — Loss Function Comparison

The following loss functions are compared:

1. Binary Cross Entropy
2. Mean Squared Error
3. Weighted Binary Cross Entropy
4. Focal Loss

The purpose is to investigate how the loss function affects classification,
especially under class imbalance.

In [ ]:
weight_0 = class_weights[0]
weight_1 = class_weights[1]

print("Class 0 weight:", weight_0)
print("Class 1 weight:", weight_1)

In [ ]:
def weighted_binary_crossentropy(y_true, y_pred):
    
    y_true = tf.cast(y_true, tf.float32)
    
    bce = keras.backend.binary_crossentropy(
        y_true,
        y_pred
    )
    
    weights = (
        y_true * weight_1
        + (1.0 - y_true) * weight_0
    )
    
    return tf.reduce_mean(
        weights * bce
    )

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    
    def loss_function(y_true, y_pred):
        
        y_true = tf.cast(y_true, tf.float32)
        
        epsilon = tf.keras.backend.epsilon()
        
        y_pred = tf.clip_by_value(
            y_pred,
            epsilon,
            1.0 - epsilon
        )
        
        pt = (
            y_true * y_pred
            + (1 - y_true) * (1 - y_pred)
        )
        
        alpha_factor = (
            y_true * alpha
            + (1 - y_true) * (1 - alpha)
        )
        
        focal_weight = tf.pow(
            1 - pt,
            gamma
        )
        
        loss = (
            -alpha_factor
            * focal_weight
            * tf.math.log(pt)
        )
        
        return tf.reduce_mean(loss)
    
    return loss_function

In [ ]:
loss_experiments = {
    "Binary Cross Entropy": "binary_crossentropy",
    "MSE": "mse",
    "Weighted BCE": weighted_binary_crossentropy,
    "Focal Loss": focal_loss(gamma=2.0, alpha=0.25)
}

loss_results = []
loss_histories = {}

for name, loss_function in loss_experiments.items():
    
    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)
    
    model = build_ann(
        input_dim=input_dim,
        activation="relu",
        initializer="he_normal",
        optimizer="adam",
        loss=loss_function,
        batch_norm=False
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        f"Loss - {name}"
    )
    
    loss_results.append(result)
    loss_histories[name] = history

In [ ]:
loss_results_df = pd.DataFrame(
    loss_results
)

display(
    loss_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(10, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

for name, history in loss_histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=name,
        linewidth=2
    )

plt.title(
    "Loss Function Comparison",
    color="#FFFFFF"
)

plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Validation Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Loss Function Comparison

Validation accuracy is compared for different loss functions.

# Task 6 — Batch Normalization

Two models are compared:

1. ANN without Batch Normalization
2. ANN with Batch Normalization

The objective is to study whether Batch Normalization improves
training stability and convergence.

In [ ]:
batchnorm_results = []
batchnorm_histories = {}

for use_bn in [False, True]:
    
    name = "With BatchNorm" if use_bn else "Without BatchNorm"
    
    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)
    
    model = build_ann(
        input_dim=input_dim,
        activation="relu",
        initializer="he_normal",
        optimizer="adam",
        loss="binary_crossentropy",
        batch_norm=use_bn
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        name
    )
    
    batchnorm_results.append(result)
    batchnorm_histories[name] = history

In [ ]:
batchnorm_results_df = pd.DataFrame(
    batchnorm_results
)

display(
    batchnorm_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(9, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

for name, history in batchnorm_histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=name,
        linewidth=2
    )

plt.title(
    "Batch Normalization Comparison",
    color="#FFFFFF"
)

plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Validation Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Batch Normalization Comparison

Validation accuracy is compared with and without Batch Normalization.

# Task 7 — Optimizer Comparison

The following optimizers are compared:

- SGD
- SGD with Momentum
- RMSprop
- Adam

The goal is to determine which optimizer provides the best
convergence and classification performance.

In [ ]:
optimizers = {
    "SGD": keras.optimizers.SGD(
        learning_rate=0.01
    ),
    
    "SGD + Momentum": keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9
    ),
    
    "RMSprop": keras.optimizers.RMSprop(
        learning_rate=0.001
    ),
    
    "Adam": keras.optimizers.Adam(
        learning_rate=0.001
    )
}

In [ ]:
optimizer_results = []
optimizer_histories = {}

for name, optimizer in optimizers.items():
    
    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)
    
    model = build_ann(
        input_dim=input_dim,
        activation="relu",
        initializer="he_normal",
        optimizer=optimizer,
        loss="binary_crossentropy",
        batch_norm=True
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        f"Optimizer - {name}"
    )
    
    optimizer_results.append(result)
    optimizer_histories[name] = history

In [ ]:
optimizer_results_df = pd.DataFrame(
    optimizer_results
)

display(
    optimizer_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(10, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

for name, history in optimizer_histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=name,
        linewidth=2
    )

plt.title(
    "Optimizer Convergence Comparison",
    color="#FFFFFF"
)

plt.xlabel("Epoch", color="#FFFFFF")
plt.ylabel("Validation Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

### Optimizer Comparison

Validation accuracy is compared for different optimizers.

In [ ]:
learning_rates = [
    0.0001,
    0.001,
    0.01,
    0.1
]

lr_results = []

for lr in learning_rates:
    
    print("Testing learning rate:", lr)
    
    model = build_ann(
        input_dim=input_dim,
        activation="relu",
        initializer="he_normal",
        optimizer=keras.optimizers.SGD(
            learning_rate=lr
        ),
        loss="binary_crossentropy",
        batch_norm=True
    )
    
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=30,
        batch_size=256,
        verbose=0
    )
    
    result, _, _ = evaluate_model(
        model,
        X_test_scaled,
        y_test,
        f"SGD LR={lr}"
    )
    
    result["Learning Rate"] = lr
    lr_results.append(result)

lr_results_df = pd.DataFrame(lr_results)

display(
    lr_results_df.round(4)
)

In [ ]:
plt.figure(figsize=(8, 5), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

plt.plot(
    lr_results_df["Learning Rate"],
    lr_results_df["Accuracy"],
    marker="o",
    linewidth=2
)

plt.xscale("log")

plt.title(
    "Learning Rate Sensitivity — SGD",
    color="#FFFFFF"
)

plt.xlabel("Learning Rate", color="#FFFFFF")
plt.ylabel("Test Accuracy", color="#FFFFFF")

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.show()

### Learning Rate Sensitivity

Model performance is compared across different learning rates.

In [ ]:
all_results = pd.concat(
    [
        pd.DataFrame([baseline_result]),
        activation_results_df,
        initialization_results_df,
        loss_results_df,
        batchnorm_results_df,
        optimizer_results_df
    ],
    ignore_index=True
)

display(
    all_results.round(4)
)

In [ ]:
final_ranking = all_results.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

display(
    final_ranking.round(4)
)

In [ ]:
best_model_name = final_ranking.loc[0, "Model"]

print("Best model based on F1-score:")
print(best_model_name)

print("\nPerformance:")
display(
    final_ranking.iloc[[0]].round(4)
)

In [ ]:
plt.figure(figsize=(12, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

plt.bar(
    final_ranking["Model"],
    final_ranking["F1"],
    color="skyblue"
)

plt.title(
    "Complete Model Comparison — F1 Score",
    color="#FFFFFF"
)

plt.xlabel("Model", color="#FFFFFF")
plt.ylabel("F1 Score", color="#FFFFFF")

plt.xticks(
    rotation=60,
    ha="right",
    color="#FFFFFF"
)

plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(
    y_test,
    baseline_prob
)

plt.figure(figsize=(7, 6), facecolor="#070808")

ax = plt.gca()
ax.set_facecolor("#070808")

plt.plot(
    fpr,
    tpr,
    label=f"Baseline ANN AUC = {baseline_result['ROC-AUC']:.3f}",
    linewidth=2
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5
)

plt.title(
    "ROC Curve — Baseline ANN",
    color="#FFFFFF"
)

plt.xlabel(
    "False Positive Rate",
    color="#FFFFFF"
)

plt.ylabel(
    "True Positive Rate",
    color="#FFFFFF"
)

plt.xticks(color="#FFFFFF")
plt.yticks(color="#FFFFFF")

for spine in ax.spines.values():
    spine.set_color("#FFFFFF")

ax.tick_params(colors="#FFFFFF")

plt.legend(
    facecolor="#070808",
    edgecolor="#FFFFFF",
    labelcolor="#FFFFFF"
)

plt.show()

# Conclusion

This project investigated Adult Income prediction using an Artificial
Neural Network.

The experiments compared:

- Different activation functions
- Different weight initialization techniques
- Different loss functions
- Batch Normalization
- Different optimizers
- Different learning rates

The models were evaluated using Accuracy, Precision, Recall, F1-score
and ROC-AUC.

Because the Adult Income dataset contains class imbalance, F1-score,
Recall and ROC-AUC were considered along with accuracy.

The final model was selected based on the experimental results rather
than assuming that one technique would always be best.

The experiments demonstrate that neural-network performance depends on
the combination of architecture, activation function, initialization,
loss function, normalization and optimization strategy.

In [ ]:
# Save final comparison table
all_results.to_csv(
    "final_results.csv",
    index=False
)

# Save ranking
final_ranking.to_csv(
    "final_model_ranking.csv",
    index=False
)

print("Results saved successfully.")